# Capítulo 7: O que é Aprendizado Estatístico

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 2 de James et al. (2023).

🎛️ [**Flexibilidade e erro ↗**](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/apoio/flexibilidade-e-erro.html): a página interativa da aula. Mova a flexibilidade de um ajuste e veja o MSE de treino cair enquanto o de teste desenha a curva em U, com o viés ao quadrado, a variância e $\mathrm{Var}(\varepsilon)$ somando o erro de teste a cada passo. Troque a $f$ verdadeira para ver o mínimo mudar de lugar.

> Todos os modelos estão errados, mas alguns são úteis.
>
> — George E. P. Box

Uma tabela de dados, com colunas que se medem e uma coluna que se quer prever: o que significa estimar a relação entre elas, o que essa estimativa promete, o que ela não pode prometer, e como saber se ela é boa? As respostas a essas perguntas são o que o resto do livro chama de aprendizado estatístico.

A seção 7.1 é opcional. Ela apresenta o `array` do `numpy`, o objeto em que um estimador do `scikit-learn` devolve seus resultados (os coeficientes, as previsões, as probabilidades): como ele se fatia sem copiar, como se filtra com uma máscara booleana e o que significa reduzir "ao longo de um eixo". As seções seguintes não dependem dela, e explicam cada função no ponto em que ela aparece.

O capítulo começa de fato na seção 7.2, com o gasto em propaganda de `Advertising` e a renda simulada de `Income1` e `Income2`. Ela define preditores, resposta e a função $f$ que liga os dois, e pergunta o que significa estimar $f$ a partir de dados. A seção 7.3 separa duas maneiras de estimar: assumir uma forma para $f$ ou deixar os dados decidirem. A 7.4 mostra por que um método mais flexível é mais difícil de interpretar, e por que ele nem sempre prevê melhor.

As três últimas seções separam o aprendizado em famílias e o ajuste em graus. A 7.5 distingue supervisionado de não supervisionado, conforme exista ou não uma resposta $Y$ para comparar com a previsão, e regressão de classificação, conforme o tipo de $Y$. A 7.6 mede o erro quadrático médio onde ele importa, no teste e não no treino, e explica a curva em U pelo compromisso entre viés e variância. A 7.7 faz o mesmo para quando $Y$ é uma classe, com a taxa de erro no lugar do MSE e o classificador de Bayes como o piso contra o qual todo classificador se mede, a começar pelo *k*-NN.

Ao final deste capítulo, você será capaz de:

- Formular a relação entre preditores e resposta como $Y = f(X) + \varepsilon$, distinguir predição de inferência e separar o erro de uma previsão em parcela redutível e irredutível
- Diferenciar o caminho paramétrico do não paramétrico para estimar $f$, e relacionar a rigidez de um e a flexibilidade do outro à quantidade de dados que cada um exige
- Explicar o compromisso entre precisão e interpretabilidade, e justificar quando escolher de propósito um método menos flexível
- Classificar um problema como supervisionado ou não supervisionado e, dentro do supervisionado, como regressão ou classificação
- Medir a qualidade de um ajuste pelo MSE de teste, e não pelo de treino, e reconhecer na curva em U o compromisso entre viés e variância
- Usar a taxa de erro como o análogo do MSE na classificação, explicar por que o classificador de Bayes é ótimo e inatingível, e como o *k*-NN se aproxima dele conforme varia $k$
- (Seção opcional 7.1) Manipular arrays do `numpy`: fatiar, filtrar com máscara booleana, reduzir ao longo de um eixo, sortear com semente e converter um `DataFrame` em `ndarray`

## Seções

| Seção | Tópico |
|---|---|
| [7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-o-array.html) | O Array (opcional) |
| [7.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/02-estimar-f.html) | Estimar f: Predição e Inferência |
| [7.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-parametrico-e-nao-parametrico.html) | Paramétrico e Não Paramétrico |
| [7.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-precisao-contra-interpretabilidade.html) | Precisão contra Interpretabilidade |
| [7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-supervisionado-e-nao-supervisionado.html) | Supervisionado e Não Supervisionado |
| [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-qualidade-do-ajuste-e-vies-variancia.html) | Qualidade do Ajuste e o Compromisso Viés-Variância |
| [7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-classificacao-e-o-classificador-de-bayes.html) | Classificação e o Classificador de Bayes |

## O Array

> **📌 Nota**
>
> Esta seção corresponde à seção 2.3 de James et al. (2023).

Os estimadores do `scikit-learn` devolvem seus resultados em arrays do `numpy`: os coeficientes ajustados, as previsões e as probabilidades chegam como `ndarray`. O array se parece com uma lista Python, mas guarda um tipo só, fatia sem copiar e faz conta sobre todos os elementos de uma vez, sem laço.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")

### O array, e o tipo único que ele impõe

Um `np.array` nasce de uma lista, mas não herda a flexibilidade dela: uma lista Python guarda qualquer mistura de tipos, e um array guarda só um.

In [ ]:
quartos = np.array([2, 3, 1, 4, 2])
quartos, quartos.dtype

> **🔧 Função**
>
> **`np.array(lista)`**: cria um array a partir de uma lista (ou de uma lista de listas, para duas dimensões).
>
> **`arr.dtype`**: o tipo único de todos os elementos do array.

`quartos` guarda os mesmos cinco números da lista, e ganha um atributo que a lista não tem: `dtype`, aqui `int64`. Basta um elemento vir com uma casa decimal para promover o array inteiro:

In [ ]:
misto = np.array([1, 2, 3.0])
misto.dtype

O resultado é `float64` para o array todo, não uma mistura de `int64` e `float64` elemento a elemento. Os dois primeiros números também viraram ponto flutuante, mesmo tendo entrado como inteiros. Uma lista de listas vira um array de duas dimensões, e `shape` guarda o formato:

In [ ]:
matriz = np.array([[1, 2], [3, 4], [5, 6]])
matriz.shape, matriz.dtype

> **🔧 Função**
>
> **`arr.shape`**: o formato do array, uma tupla com o tamanho de cada dimensão. `(3, 2)` são três linhas e duas colunas; `(5,)` é uma dimensão só, com cinco elementos.

Três linhas e duas colunas, e de novo um `dtype` só para o array inteiro.

### Fatiar sem copiar: a vista

Fatiar um array usa a mesma notação `[início:fim]` de uma lista Python, mas o resultado se comporta de outro jeito.

In [ ]:
original = np.array([10, 20, 30, 40, 50])
fatia = original[1:3]
fatia[0] = 999
original

O que aconteceu com `original` quando só `fatia` foi alterada? Mudou também. A fatia é outra janela sobre o mesmo bloco de memória, e não uma cópia dos números. Quem chega das listas cai nessa armadilha primeiro, porque lá `lista[1:3]` sempre devolve uma lista nova, sem ligação nenhuma com a original. Quando o array de origem precisa continuar intocado, a fatia pede `.copy()`:

In [ ]:
original2 = np.array([10, 20, 30, 40, 50])
copia = original2[1:3].copy()
copia[0] = 999
original2

> **🔧 Função**
>
> **`arr.copy()`**: devolve um array novo, num bloco de memória separado, com os mesmos valores.

Desta vez `original2` sai como entrou.

> **🔷 Conceito**
>
> Fatia (`x[1:3]`) devolve uma **vista**: o mesmo bloco de memória, enxergado por outra janela. Indexação por **máscara booleana** ou por **lista de posições** (`x[[0, 2]]`) sempre devolve uma **cópia**. `.copy()` força uma cópia em qualquer caso, quando o array original precisa ficar fora de alcance.

### A máscara booleana no lugar do laço

Uma comparação entre um array e um número acontece elemento a elemento, e devolve um array de `True`/`False` do mesmo tamanho.

In [ ]:
valores = np.array([-3, 5, -1, 8, 0, -7, 2])
mascara = valores > 0
mascara

Indexar o array original com esse array de booleanos, `valores[mascara]`, filtra os elementos onde a máscara vale `True`:

In [ ]:
valores[mascara]

Um laço `for` com um `if` dentro produziria o mesmo resultado, elemento a elemento. A máscara descreve a condição uma vez, e o `numpy` a aplica sobre o array inteiro.

### A forma de uma redução: `axis`

Somar os elementos de uma matriz aceita um argumento que muda o que "somar" quer dizer. `axis=0` percorre as linhas, coluna por coluna; `axis=1` percorre as colunas, linha por linha. Sobre uma matriz com aluguel e área de quatro imóveis:

In [ ]:
precos = np.array([
    [1200.0, 65.0],
    [800.0, 42.0],
    [2100.0, 98.0],
    [950.0, 55.0],
])
precos.shape

Quatro linhas, duas colunas. Somando ao longo de `axis=0`:

In [ ]:
soma_colunas = precos.sum(axis=0)
soma_colunas.shape, soma_colunas

> **🔧 Função**
>
> **`arr.sum(axis)`** e **`arr.mean(axis)`**: somam ou tiram a média dos elementos. Sem `axis`, reduzem o array inteiro a um número.
>
> - `axis`: o eixo que desaparece na redução. Com `0`, sobra um resultado por coluna; com `1`, um por linha.

Um total por **coluna**, e por isso a forma `(2,)`: 5.050,0 de aluguel somado e 260,0 de área somada, os quatro imóveis colapsados numa soma cada. Somando ao longo de `axis=1`:

In [ ]:
soma_linhas = precos.sum(axis=1)
soma_linhas.shape, soma_linhas

Agora são quatro números, um por **linha**: aluguel mais área de cada imóvel, uma soma que aqui só mostra a forma do resultado, porque reais e metros quadrados não se somam. A confusão mais comum de quem começa com `axis` mora aqui. O número que ele nomeia é o eixo que **desaparece** na redução, e não o eixo que sobra.

`mean` segue a mesma regra, trocando soma por média:

In [ ]:
media_colunas = precos.mean(axis=0)
media_colunas.shape, media_colunas

1.262,5 de aluguel médio e 65,0 de área média, uma média por coluna.

In [ ]:
media_linhas = precos.mean(axis=1)
media_linhas.shape, media_linhas

E aqui, um número por imóvel, com a mesma ressalva de unidade. A média por coluna é a que mais volta nos capítulos seguintes: é com ela que se padroniza uma variável.

### Sorteando com semente: o `rng`

Um sorteio reprodutível precisa de um gerador com semente fixa e explícita. Sem semente, cada renderização da página sortearia números diferentes, e as figuras que dependem deles mudariam a cada vez sem que o texto ao redor mudasse junto.

In [ ]:
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
np.array_equal(rng_a.normal(size=3), rng_b.normal(size=3))

> **🔧 Função**
>
> **`np.random.default_rng(semente)`**: cria um gerador de números aleatórios. Dois geradores com a mesma semente sorteiam a mesma sequência.
>
> **`np.array_equal(a, b)`**: `True` se os dois arrays têm o mesmo formato e os mesmos valores.

Duas instâncias criadas com a mesma semente sorteiam exatamente a mesma sequência. É essa garantia que faz `rng = np.random.default_rng(7)` valer como semente fixa do capítulo inteiro:

In [ ]:
rng = np.random.default_rng(7)
amostra = rng.normal(size=5)
amostra

`rng.normal` sorteia de uma normal padrão. `rng.choice` sorteia entre valores dados, cada um com a mesma chance por padrão:

In [ ]:
rng.choice(["sim", "não"], size=5)

> **🔧 Função**
>
> **`rng.normal(loc, scale, size)`**: sorteia `size` valores de uma normal com média `loc` e desvio padrão `scale` (por padrão, 0 e 1).
>
> **`rng.choice(valores, size)`**: sorteia `size` elementos de `valores`, com reposição.

A figura a seguir junta as duas coisas desta seção: coordenadas sorteadas com `rng.normal`, e uma máscara booleana que decide a cor de cada ponto.

In [ ]:
# Figura: Duzentos pontos sorteados com `rng.normal`, coloridos por uma máscara booleana sobre a distância à origem
x = rng.normal(size=200)
y = rng.normal(size=200)
distancia = np.sqrt(x**2 + y**2)
mascara_fig = distancia > 1.5

fig, ax = plt.subplots()
ax.scatter(x[~mascara_fig], y[~mascara_fig], label="até 1,5 da origem")
ax.scatter(x[mascara_fig], y[mascara_fig], label="além de 1,5 da origem")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.sqrt(arr)`**: a raiz quadrada de cada elemento. Operadores como `**` e `+` também agem elemento a elemento.
>
> **`~mascara`**: inverte uma máscara, trocando `True` por `False` e vice-versa.
>
> **`ax.set_aspect("equal")`**: a mesma escala nos dois eixos, para que um círculo apareça como círculo.

In [ ]:
int(mascara_fig.sum()), int((~mascara_fig).sum())

Somar uma máscara conta quantos `True` ela tem, porque, numa soma, `True` vale 1. 55 dos 200 pontos ficam a mais de 1,5 da origem, e os outros 145 ficam mais perto. É a mesma máscara que filtrou `valores` mais acima, agora escolhendo a cor de um gráfico.

### Do `DataFrame` para o array

Uma tabela de `pandas` guarda, além dos números, o nome de cada coluna e o índice de cada linha. Em `alugueis`, quatro colunas numéricas descrevem cada imóvel (os preditores, `X`) e o aluguel é o que se quer prever (a resposta, `y`):

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"])
X = alugueis[["area_m2", "quartos", "banheiros", "vagas"]]
y = alugueis["aluguel"]
type(X), X.shape, type(y), y.shape

`.to_numpy()` devolve o que está por baixo de cada um: só os números, sem rótulo de coluna nem índice.

In [ ]:
X_array = X.to_numpy()
y_array = y.to_numpy()
type(X_array), X_array.shape, X_array.dtype, type(y_array), y_array.shape, y_array.dtype

> **🔧 Função**
>
> **`df.to_numpy()`** e **`serie.to_numpy()`**: devolvem os valores de uma tabela ou de uma coluna como `ndarray`, sem nomes de coluna nem índice.

As formas continuam `(10692, 4)` e `(10692,)`, mas o `DataFrame` e a `Series` viraram `ndarray`, e com o tipo foram embora os nomes de coluna e o índice. A primeira linha de `X_array`,

In [ ]:
X_array[0]

chega como quatro números soltos (70, 2, 1, 1), sem dizer qual número é a área, quais são os quartos, os banheiros e as vagas. Só a ordem em que as colunas foram escolhidas guarda esse significado. É nesse formato, sem nome de coluna, que um estimador do `scikit-learn` faz as contas e devolve os resultados.

## Estimar f: Predição e Inferência

> **📌 Nota**
>
> Esta seção corresponde às seções 2.1 e 2.1.1 de James et al. (2023).

Uma empresa vende o mesmo produto em duzentos mercados e, em cada um, decidiu quanto gastar com propaganda na TV, no rádio e no jornal. Existe relação entre o que ela gasta e o que vende? E, se existe, dá para usá-la para prever as vendas de um orçamento novo, ou para decidir em qual mídia vale a pena investir?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")

### Observações, preditores e resposta

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
propaganda.head()

Cada linha é um mercado. `tv`, `radio` e `jornal` são o gasto com cada mídia, em milhares de dólares, e `vendas` é o volume vendido, em milhares de unidades.

As colunas se dividem em dois papéis. O que se mede e, em alguma medida, se controla (o gasto com cada mídia) são os **preditores**, também chamados de variáveis de entrada, variáveis independentes ou *features*. O que se quer prever ou explicar (as vendas) é a **resposta**, também chamada de variável de saída ou variável dependente. Em notação, os preditores são $X_1$ = `tv`, $X_2$ = `radio` e $X_3$ = `jornal`, e juntos formam $X = (X_1, X_2, X_3)$; a resposta é $Y$ = `vendas`. Com $p$ preditores quaisquer, escreve-se $X = (X_1, X_2, \dots, X_p)$.

No código, os mesmos dois papéis viram dois objetos: `X`, uma tabela só com as colunas dos preditores (colchete duplo, porque são várias colunas), e `y`, a coluna da resposta (colchete simples).

In [ ]:
X = propaganda[["tv", "radio", "jornal"]]
y = propaganda["vendas"]
X.shape, y.shape

`X` tem 200 linhas e 3 colunas; `y`, as mesmas 200 linhas. O número de observações se chama $n$ e o de preditores, $p$: aqui, $n = 200$ e $p = 3$. A linha $i$ de `X` e a posição $i$ de `y` descrevem o mesmo mercado, e esse par é a observação $i$, escrita $(x_i, y_i)$ para $i = 1, \dots, n$.

Antes de qualquer fórmula, vale olhar a relação que se quer descrever: vendas contra o gasto em cada mídia, um painel por preditor.

In [ ]:
# Figura: Vendas (milhares de unidades) contra o gasto em cada mídia (milhares de dólares), nos 200 mercados de `Advertising`. Cada ponto é um mercado.
fig, eixos = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True)
for ax, midia in zip(eixos, ["tv", "radio", "jornal"]):
    ax.scatter(propaganda[midia], propaganda["vendas"], s=12, alpha=0.6, clip_on=False)
    ax.set_xlabel(midia)
    ax.set_xlim(left=0)
eixos[0].set_ylabel("vendas")
plt.tight_layout()
plt.show()

Com `sharey=True`, os três painéis usam a mesma escala vertical, para que as vendas se comparem entre eles; `set_xlim(left=0)` começa cada eixo horizontal em zero, porque não existe gasto negativo. Em TV, os pontos se alinham numa faixa que sobe; em jornal, espalham-se quase sem forma. A correlação mede o quanto cada nuvem se aproxima de uma reta:

In [ ]:
propaganda.corr()["vendas"].round(2)

> **🔧 Função**
>
> **`df.corr()`**: a correlação de Pearson de cada coluna com cada outra, numa tabela quadrada. `["vendas"]` separa a coluna das correlações com `vendas`.

0,78 com TV, 0,58 com rádio e 0,23 com jornal. Em nenhuma das três a nuvem é uma linha: mercados com o mesmo gasto em TV vendem quantidades bem diferentes.

### A formulação: $Y = f(X) + \varepsilon$

O ponto de partida do aprendizado supervisionado é supor que existe uma relação entre $X$ e $Y$ e que ela se escreve assim:

$$Y = f(X) + \varepsilon$$

$f$ é uma função fixa, mas desconhecida, dos preditores. $f(x)$ é a venda média de todos os mercados que investem $x$, isto é, a parte das vendas que o gasto com propaganda explica. $\varepsilon$ é o **erro**: o quanto um mercado específico se afasta dessa média. Nele cabe tudo o que influencia as vendas e não está em $X$, como a época do ano, um concorrente que baixou o preço, o próprio ruído de medir as vendas. Supõe-se que $\varepsilon$ seja independente de $X$ e tenha média zero, de modo que não haja tendência de errar para cima nem para baixo.

**Aprendizado estatístico** é o conjunto de métodos para estimar $f$ a partir dos dados.

### Estimar $f$: $\hat f$ e $\hat Y$

$f$ nunca aparece nos dados. O que se tem são as $n$ observações $(x_i, y_i)$, cada uma com seu $\varepsilon$ misturado. Estimar $f$ é usar essas observações para construir uma função $\hat f$ (lê-se "f chapéu") que chegue perto de $f$. Com ela, a previsão para um $X$ qualquer é

$$\hat Y = \hat f(X)$$

e o chapéu marca, nos dois símbolos, o que foi estimado a partir dos dados.

Como seria uma $\hat f$ concreta? `Income1` ajuda a ver, porque tem um preditor só e cabe num gráfico:

In [ ]:
estudo_renda = pd.read_csv("dados/Income1.csv").sort_values("escolaridade")
estudo_renda.shape, estudo_renda.columns.tolist()

Trinta pessoas, com anos de estudo (`escolaridade`, o $X$) e renda em milhares de dólares (`renda`, o $Y$). Os pontos foram simulados a partir de uma $f$ conhecida mais um ruído, mas essa $f$ não vem junto com os dados, e aqui ela fica desconhecida, como numa situação real.

Uma estimativa simples: com as pessoas em ordem de escolaridade, prever a renda de cada uma como a média da renda dela e das seis pessoas imediatamente antes e depois. Juntando essas médias, sai uma curva, e essa curva é uma $\hat f$.

In [ ]:
# Figura: Renda contra anos de estudo em `Income1`. A curva é uma estimativa de f, a média móvel das rendas; cada segmento liga uma pessoa observada à previsão da curva e mede o resíduo dela.
educacao = estudo_renda["escolaridade"]
renda = estudo_renda["renda"]
f_chapeu = renda.rolling(window=13, center=True, min_periods=1).mean()

fig, ax = plt.subplots()
ax.vlines(educacao, renda, f_chapeu, color="C1", linewidth=1)
ax.plot(educacao, f_chapeu, color="C0", linewidth=2, label="$\\hat f$ (média móvel)")
ax.scatter(educacao, renda, color="C2", zorder=3, label="observado")
ax.set_xlabel("anos de estudo")
ax.set_ylabel("renda (milhares de dólares)")
ax.legend()
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`serie.rolling(window, center, min_periods).mean()`**: a média móvel, isto é, para cada posição, a média de uma janela de valores vizinhos.
>
> - `window`: quantos valores entram em cada janela, aqui 13.
> - `center=True`: a janela fica centrada na posição, com seis vizinhos de cada lado.
> - `min_periods=1`: nas pontas, onde a janela não cabe inteira, usa os valores que houver.
>
> **`ax.vlines(x, ymin, ymax)`**: desenha um segmento vertical em cada `x`, de `ymin` a `ymax`.

Cada segmento é um **resíduo**, $y_i - \hat y_i$: a distância entre o que a pessoa ganha e o que a curva prevê para os anos de estudo dela.

In [ ]:
residuo = renda - f_chapeu
acima = int((residuo > 0).sum())
abaixo = int((residuo < 0).sum())
media = round(float(residuo.mean()), 2)
acima, abaixo, media

Dezesseis pessoas ficam acima da curva e quatorze abaixo, e a média dos resíduos é 0,10. Sai perto de zero porque a curva é, ela mesma, uma média das rendas: dentro de cada janela, as distâncias para cima e para baixo quase se compensam. Não sai exatamente zero porque nas pontas a janela encolhe, e as rendas das pontas entram em menos médias que as do meio.

Esses segmentos misturam duas coisas. Parte deles é $\varepsilon$, o quanto cada pessoa se afasta da renda média de quem estudou o mesmo tanto. A outra parte é o erro da própria curva, a distância entre $\hat f$ e $f$. O gráfico mostra a soma das duas e não tem como separá-las, porque $f$ não está nos dados.

### Prever ou explicar: duas perguntas diferentes

O que se faz com $\hat f$ depende da pergunta. "Quanto vou vender com este orçamento?" é uma pergunta de **predição**: importa que $\hat Y$ saia perto de $Y$, e a forma de $\hat f$ pode ficar escondida. Um modelo tratado como caixa-preta serve, desde que a previsão acerte.

"Quais mídias estão associadas a vendas maiores, e quanto?" é uma pergunta de **inferência**. Aqui a forma de $f$ é o que se quer entender: quais preditores têm associação com a resposta, se ela é positiva ou negativa, se um gasto pequeno em jornal já esgota o que jornal tem a oferecer. Aqui, "inferência" quer dizer *entender* a relação entre $X$ e $Y$, e não calcular erro-padrão ou valor-p.

As duas perguntas pedem modelos diferentes, mesmo sobre o mesmo par $(X, Y)$. Uma caixa-preta pode prever as vendas com folga e não dizer nada sobre qual mídia cortar num aperto de orçamento. Um modelo simples o bastante para se ler a associação de cada mídia pode prever pior que um mais flexível.

### Erro redutível e irredutível

Por que nenhuma previsão acerta sempre, nem com o melhor método? Há duas fontes de erro, e só uma delas depende do método.

A primeira é a distância entre $\hat f$ e $f$, o erro **redutível**: um método mais adequado ou mais observações, com os mesmos preditores, podem encolhê-la. A segunda é $\varepsilon$. Mesmo que $\hat f$ fosse igual a $f$ em todo ponto, a previsão $\hat Y = f(X)$ ainda erraria, porque $Y$ também depende de $\varepsilon$, que não se prevê a partir de $X$. Esse é o erro **irredutível**.

A conta que separa as duas parcelas fixa uma estimativa $\hat f$ e um valor de $X$, de modo que a única coisa aleatória que sobra é $\varepsilon$. Nessa situação, o erro quadrático esperado da previsão é

$$
\mathrm{E}\left[(Y - \hat{Y})^2\right] = \underbrace{\left[f(X) - \hat{f}(X)\right]^2}_{\text{redutível}} + \underbrace{\mathrm{Var}(\varepsilon)}_{\text{irredutível}}
$$

O primeiro termo é o que um método melhor reduz. O segundo não depende do método, e funciona como um piso: nenhum ajuste, por melhor que seja, põe o erro esperado abaixo de $\mathrm{Var}(\varepsilon)$. Medir um preditor novo (o preço do concorrente, por exemplo) muda $X$, e com ele muda o que conta como $\varepsilon$; mas esse já é outro problema, com outro piso.

Os segmentos da figura de `Income1` misturam as duas parcelas. Uma curva que passasse exatamente por cada ponto zeraria todos eles, sem ter chegado mais perto de $f$: teria só copiado o $\varepsilon$ de cada pessoa. Para saber se uma curva se aproximou de $f$ ou só copiou o ruído, o erro precisa ser medido em observações que não foram usadas para construí-la. Esse erro ainda soma as duas parcelas, mas o irredutível é o mesmo para qualquer método, e por isso comparar esse erro entre métodos compara o redutível. É o que a seção 7.6 faz.

## Paramétrico e Não Paramétrico

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1.2 de James et al. (2023).

Como se estima $f$ a partir só do que se observa, sem acesso à função que gerou os dados? Há duas famílias de resposta, e o que as separa é uma escolha feita antes de olhar qualquer ponto: assumir uma forma para $f$, ou deixar que os dados decidam a forma.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

plt.style.use("estilo-figuras.mplstyle")

### O caminho paramétrico: assumir uma forma

O caminho paramétrico tem dois passos. Primeiro, escolhe-se uma forma para $f$ que se escreva com um número fixo e pequeno de parâmetros, como uma reta ou um plano. Depois, usam-se os dados para encontrar os valores desses parâmetros. O problema de estimar uma função inteira, livre para ter qualquer formato, vira o problema bem mais simples de estimar poucos números.

A vantagem é que poucos parâmetros pedem poucos dados, porque cada observação ajuda a fixar todos eles ao mesmo tempo. O risco está na mesma decisão. Se a forma escolhida for muito diferente da $f$ verdadeira, nenhuma quantidade de dados conserta: o erro já está na forma, antes de o primeiro ponto entrar na conta.

`Income2` dá o exemplo, com a renda de trinta pessoas contra dois preditores.

In [ ]:
renda2 = pd.read_csv("dados/Income2.csv")
renda2.shape, renda2.columns.tolist()

`escolaridade` (anos de estudo) e `senioridade` são os preditores, e `renda`, em milhares de dólares, é a resposta. Antes de escolher uma forma, vale olhar a renda contra cada preditor:

In [ ]:
# Figura: Renda contra escolaridade e contra senioridade, nas trinta pessoas de `Income2`.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
ax1.scatter(renda2["escolaridade"], renda2["renda"])
ax1.set_xlabel("escolaridade")
ax1.set_ylabel("renda")
ax2.scatter(renda2["senioridade"], renda2["renda"])
ax2.set_xlabel("senioridade")
plt.tight_layout()
plt.show()

A renda cresce com os dois preditores, com mais dispersão na senioridade. Com dois preditores, a forma paramétrica mais simples que acompanha esse crescimento é um plano, o **modelo linear**:

$$
\text{renda} \approx \beta_0 + \beta_1 \times \text{escolaridade} + \beta_2 \times \text{senioridade}
$$

Assumida essa forma, estimar $f$ é encontrar três números: o intercepto $\beta_0$ e os coeficientes $\beta_1$ e $\beta_2$. O jeito mais comum de encontrá-los é o de **mínimos quadrados**, que escolhe os valores que tornam menor a soma dos quadrados das distâncias entre cada renda observada e a renda que o plano prevê. O capítulo 8 trata desse método em detalhe; aqui basta saber que o `scikit-learn` faz a conta.

In [ ]:
X = renda2[["escolaridade", "senioridade"]]
y = renda2["renda"]

plano = LinearRegression().fit(X, y)
[round(float(coeficiente), 3) for coeficiente in plano.coef_], round(float(plano.intercept_), 2)

> **🔧 Função**
>
> **`LinearRegression()`**: cria um modelo linear ainda sem parâmetros estimados.
>
> **`modelo.fit(X, y)`**: estima os parâmetros a partir dos preditores `X` (uma tabela, uma coluna por preditor) e da resposta `y` (uma coluna, com uma linha por observação). Devolve o próprio modelo, já ajustado.
>
> **`modelo.coef_`** e **`modelo.intercept_`**: os coeficientes estimados $\hat\beta_1, \dots, \hat\beta_p$, na ordem das colunas de `X`, e o intercepto estimado $\hat\beta_0$.

O plano ajustado tem $\hat\beta_1 = 5{,}896$ para escolaridade, $\hat\beta_2 = 0{,}173$ para senioridade e intercepto $\hat\beta_0 = -50{,}09$. Três números descrevem a superfície inteira. Com trinta pontos ou com trinta mil, o modelo teria os mesmos três parâmetros; mais dados deixariam a estimativa mais precisa, mas não mudariam quantos são.

### O caminho não paramétrico: deixar os dados decidirem

O caminho não paramétrico não assume forma nenhuma para $f$. Em vez de resumir a relação em poucos números, deixa que a vizinhança de cada ponto decida o valor previsto ali. O exemplo mais simples é o de ***k* vizinhos mais próximos** (*k*-NN): para prever a renda de alguém com certa escolaridade e senioridade, procuram-se as $k$ pessoas mais parecidas nos dados e tira-se a média das rendas delas. Nada na conta supõe que a relação seja um plano, uma curva suave ou qualquer forma com nome.

A vantagem é acertar formas que o plano erraria de saída. O custo é precisar de muito mais dados para essa liberdade compensar. E, levada longe demais, a flexibilidade passa a ajustar o ruído das trinta pessoas desta amostra em vez da relação verdadeira.

Antes de ajustar, há um detalhe que o *k*-NN cobra. "Mais parecidas" quer dizer "mais próximas" numa distância calculada com os dois preditores, e a escala de cada um pesa nessa conta:

In [ ]:
X.agg(["mean", "std"]).round(1)

Sem `groupby` antes, `agg` resume a tabela inteira, coluna a coluna. Escolaridade tem média 16,4 e desvio padrão 3,8; senioridade, média 93,9 e desvio padrão 55,7. Com escalas tão diferentes, a distância seria dominada pela senioridade, e a escolaridade quase não pesaria na escolha dos vizinhos. A **padronização** resolve isso: de cada coluna se subtrai a média dela e se divide pelo desvio padrão dela, e as duas passam a variar na mesma escala.

In [ ]:
padronizado = (X - X.mean()) / X.std()
flexivel = KNeighborsRegressor(n_neighbors=3).fit(padronizado, y)

mse_plano = mean_squared_error(y, plano.predict(X))
mse_flexivel = mean_squared_error(y, flexivel.predict(padronizado))

erro_treino = pd.Series(
    {"plano": mse_plano, "k-NN flexível (k=3)": mse_flexivel}
).sort_values()
erro_treino.round(2)

> **🔧 Função**
>
> **`df.mean()`** e **`df.std()`**: a média e o desvio padrão de cada coluna. Em `(X - X.mean()) / X.std()`, a conta é feita coluna a coluna, cada uma com a sua média e o seu desvio.
>
> **`KNeighborsRegressor(n_neighbors)`**: cria um *k*-NN para resposta numérica, que prevê a média da resposta dos `n_neighbors` vizinhos mais próximos.
>
> **`modelo.predict(X)`**: aplica o modelo ajustado a cada linha de `X` e devolve as previsões $\hat y$.
>
> **`mean_squared_error(y, y_previsto)`**: o erro quadrático médio, $\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat y_i)^2$, a média dos quadrados das diferenças entre o observado e o previsto.

Os dois erros foram medidos sobre os mesmos trinta pontos usados no ajuste, o **erro de treino**. Nele, o *k*-NN erra menos que o plano: 36,69 de erro quadrático médio contra 46,48. Livre de forma, a superfície encosta mais nos pontos observados do que o plano consegue.

E se a flexibilidade for levada ao extremo, com um único vizinho em vez de três?

In [ ]:
memoriza = KNeighborsRegressor(n_neighbors=1).fit(padronizado, y)
round(mean_squared_error(y, memoriza.predict(padronizado)), 2)

O erro de treino cai a exatamente zero. Com um vizinho só, a renda prevista para cada pessoa do treino é a renda dela mesma, porque cada pessoa é a sua própria vizinha mais próxima, a distância zero. O modelo não aprendeu nada sobre como escolaridade e senioridade se relacionam com renda: decorou as trinta respostas que já tinha.

Um erro de treino zero é, portanto, um sinal de alerta, e não uma vitória. Ele sugere ***overfitting*** (sobreajuste): o modelo seguiu tão de perto os dados observados que passou a reproduzir o ruído daquela amostra. O diagnóstico só se confirma medindo o erro em dados que o modelo não viu durante o ajuste, que é o que a seção 7.6 faz.

> **🔷 Conceito**
>
> | | Paramétrico | Não paramétrico |
> |---|---|---|
> | Forma de $f$ | assumida antes de ver os dados (reta, plano, ...) | não assumida: os dados decidem |
> | O que se estima | um número fixo de parâmetros | a superfície inteira |
> | Quantos dados exige | poucos | muitos mais |
> | Risco principal | a forma errada, que nenhum dado corrige | seguir o ruído da amostra (*overfitting*) |

### A rigidez de um contra a flexibilidade do outro

Uma figura mostra a diferença melhor que dois números: a renda que cada ajuste prevê para toda combinação de escolaridade e senioridade, e não só para as trinta pessoas observadas. Os pontos da grade também precisam ser padronizados antes de passar pelo *k*-NN, e com a média e o desvio padrão do treino (`X.mean()`, `X.std()`), porque foi nessa escala que o modelo foi ajustado.

In [ ]:
# Figura: Renda prevista para toda combinação de escolaridade e senioridade em `Income2`, pelo plano paramétrico (esquerda) e pelo k-NN com k=3 (direita). Os pontos são as trinta pessoas observadas. As faixas retas e paralelas do plano não acompanham cada aglomerado de pontos; a superfície do k-NN se dobra ao redor deles.
grade_escolaridade = np.linspace(X["escolaridade"].min(), X["escolaridade"].max(), 60)
grade_senioridade = np.linspace(X["senioridade"].min(), X["senioridade"].max(), 60)
malha_e, malha_s = np.meshgrid(grade_escolaridade, grade_senioridade)
grade = pd.DataFrame({"escolaridade": malha_e.ravel(), "senioridade": malha_s.ravel()})
grade_padronizada = (grade - X.mean()) / X.std()

superficie_plano = plano.predict(grade).reshape(malha_e.shape)
superficie_flexivel = flexivel.predict(grade_padronizada).reshape(malha_e.shape)

niveis = np.linspace(
    min(superficie_plano.min(), superficie_flexivel.min()),
    max(superficie_plano.max(), superficie_flexivel.max()),
    13,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.contourf(
    malha_e, malha_s, superficie_plano, levels=niveis, cmap="Blues"
)
ax1.scatter(
    X["escolaridade"], X["senioridade"],
    color="#7A8894", s=18, edgecolor="white", linewidth=0.6, clip_on=False,
)
ax1.set_xlabel("escolaridade")
ax1.set_ylabel("senioridade")
ax1.set_title("paramétrico: o plano")

mapa = ax2.contourf(
    malha_e, malha_s, superficie_flexivel, levels=niveis, cmap="Blues"
)
ax2.scatter(
    X["escolaridade"], X["senioridade"],
    color="#7A8894", s=18, edgecolor="white", linewidth=0.6, clip_on=False,
)
ax2.set_xlabel("escolaridade")
ax2.set_ylabel("senioridade")
ax2.set_title("não paramétrico: k-NN (k=3)")

barra = fig.colorbar(mapa, ax=[ax1, ax2], shrink=0.85, pad=0.02)
barra.set_label("renda prevista")
barra.ax.yaxis.set_major_formatter(lambda valor, _: f"{valor:.0f}")

plt.show()

> **🔧 Função**
>
> **`np.linspace(inicio, fim, n)`**: `n` números igualmente espaçados de `inicio` a `fim`.
>
> **`np.meshgrid(a, b)`**: combina dois eixos numa grade, devolvendo duas matrizes com a coordenada horizontal e a vertical de cada ponto dela. **`.ravel()`** achata cada matriz numa sequência só, para que a grade vire uma tabela de pontos, e **`.reshape(formato)`** faz o caminho de volta, dobrando as previsões no formato da grade.
>
> **`ax.contourf(x, y, z, levels, cmap)`**: pinta o plano por faixas de valor de `z`, como num mapa de relevo. `levels` são os limites das faixas; `cmap`, a escala de cores.
>
> **`fig.colorbar(mapa, ax)`**: a barra que traduz cor em valor, ao lado dos painéis em `ax`. **`barra.ax.yaxis.set_major_formatter(funcao)`** escreve cada marca da barra com a função dada, aqui como número inteiro.

O plano nunca se curva: três números não têm como acompanhar cada solavanco dos dados, só a tendência geral. O *k*-NN se dobra ao redor de cada aglomerado de pontos, e nada o impede de seguir cada ponto isoladamente, até o extremo de erro de treino zero que o *k* = 1 mostrou. Essa flexibilidade tem um segundo preço, além dos dados que exige: um modelo que se adapta tão de perto ao que observou é mais difícil de ler. Não sobra um coeficiente para dizer "entre pessoas com a mesma senioridade, um ano a mais de escolaridade está associado a tanto a mais de renda".

## Precisão contra Interpretabilidade

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1.3 de James et al. (2023).

Um modelo flexível consegue acompanhar quase qualquer forma que os dados tenham, e paga por isso em legibilidade: não sobra nele um coeficiente para dizer quanto vale cada preditor. E essa liberdade nem sempre se converte em previsões melhores. Os dois fatos pesam na escolha do método, muitas vezes antes mesmo de se olhar os dados.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

### O compromisso: acompanhar mais, entender menos

De um lado ficam os métodos rígidos. A regressão linear que a seção 7.3 ajustou a `Income2` reduz a superfície inteira a três números, e é por ter tão pouco a ajustar que cada um deles se lê de cara: "entre pessoas com a mesma senioridade, um ano a mais de escolaridade está associado a tanto a mais de renda". Do outro lado ficam os métodos flexíveis. O *k*-NN não guarda coeficiente nenhum, só uma vizinhança que muda de ponto a ponto, e com um único vizinho ele decora as respostas que já tinha. Descrever o que um *k*-NN "aprendeu" sobre escolaridade, senioridade e renda exige desenhar a superfície inteira, e não escrever uma frase.

O padrão geral é este: quanto mais flexível o método, mais formas de $f$ ele consegue acompanhar, e menos ele se deixa resumir em algo que uma pessoa leia e repita. Acompanhar mais formas pode render previsões melhores quando a $f$ verdadeira é complicada e há dados suficientes. Mas não é garantido: um método flexível demais passa a seguir o ruído da amostra, e muitas vezes um método menos flexível prevê melhor.

### Por que escolher o menos flexível de propósito

Se um método flexível acompanha mais formas de $f$, por que alguém escolheria um menos flexível? Por dois motivos.

O primeiro é a pergunta que se está fazendo. Prever só pede que $\hat Y$ saia perto de $Y$; inferir pede entender a forma de $f$, isto é, que preditor pesa mais, em que direção e com que força. Um método flexível pode prever muito bem e não servir para essa segunda pergunta. Se a única forma de descrever o que ele faz é reproduzir o código inteiro, não há como apontar nele o que empurra a resposta para cima. Quando a pergunta é de inferência, um modelo que acerta e não se deixa explicar não responde ao que foi perguntado.

O segundo é a quantidade de dados. Com poucos dados, uma superfície livre não tem com que aprender a relação verdadeira, e o que ela aprende no lugar é o ruído daquela amostra pequena. A seção 7.3 mostrou o sintoma, um erro de treino que cai a zero; a seção 7.6 mede o custo em dados novos, onde o ajuste flexível demais erra mais que um moderado. Nesse caso, o método rígido pode ser justamente o que entrega mais precisão.

> **🔷 Conceito**
>
> | Escolher o **menos** flexível de propósito | Por quê |
> |---|---|
> | A pergunta é de inferência | um ajuste que não se lê não responde "o que explica a resposta", mesmo acertando |
> | Os dados são poucos | o método flexível não tem com que aprender a relação, e segue o ruído da amostra |

### O mapa: flexibilidade contra interpretabilidade

Postos lado a lado, alguns métodos de aprendizado estatístico (os dois já vistos, regressão linear e *k* vizinhos mais próximos, e outros que ainda vêm, como *lasso*, árvores de decisão, *bagging*, *boosting* e redes neurais) ocupam posições diferentes num mesmo eixo. Numa ponta, métodos rígidos o bastante para se ler o efeito de cada preditor; na outra, métodos livres o bastante para acompanhar quase qualquer forma que os dados tenham.

Onde ficaria cada um deles, se fosse preciso pô-los num só gráfico? O mapa a seguir não sai de dado nenhum: ninguém mediu "flexibilidade" ou "interpretabilidade" numa unidade. É uma comparação relativa entre famílias de método, útil para orientar uma escolha antes de ajustar qualquer coisa, e não um número a se citar depois.

In [ ]:
# Figura: Flexibilidade contra interpretabilidade: um mapa qualitativo, sem unidade em nenhum dos dois eixos. A posição de cada método é uma comparação relativa entre famílias, e não uma coordenada medida.
metodos = {
    "lasso": (0.08, 0.92),
    "regressão linear": (0.22, 0.80),
    "árvore de decisão": (0.45, 0.55),
    "k-vizinhos\nmais próximos": (0.58, 0.38),
    "bagging e boosting": (0.72, 0.28),
    "redes neurais": (0.92, 0.10),
}

fig, ax = plt.subplots(figsize=(7, 5.2))
for nome, (flexibilidade, interpretabilidade) in metodos.items():
    ax.scatter(flexibilidade, interpretabilidade, color="C0", s=70, zorder=3)
    ax.annotate(
        nome,
        (flexibilidade, interpretabilidade),
        textcoords="offset points",
        xytext=(8, 6),
        fontsize=9,
    )

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_xticks([0.05, 1.0])
ax.set_xticklabels(["baixa", "alta"])
ax.set_yticks([0.05, 1.0])
ax.set_yticklabels(["baixa", "alta"])
ax.set_xlabel("flexibilidade (ordem qualitativa, sem escala)")
ax.set_ylabel("interpretabilidade (ordem qualitativa, sem escala)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.annotate(texto, xy, textcoords, xytext, fontsize)`**: escreve `texto` junto ao ponto `xy`.
>
> - `textcoords="offset points"` e `xytext=(8, 6)`: o texto fica deslocado 8 pontos para a direita e 6 para cima do ponto, para não cobri-lo.
>
> **`ax.set_xticks(posicoes)`** e **`ax.set_xticklabels(rotulos)`**: onde ficam as marcas do eixo horizontal e o texto de cada uma, aqui só "baixa" e "alta" nas duas pontas. `set_yticks` e `set_yticklabels` fazem o mesmo no eixo vertical.

Lasso e regressão linear ficam no canto rígido e legível. Redes neurais, *bagging* e *boosting* ficam no canto oposto, com os métodos capazes de acompanhar as formas mais complicadas. Árvore de decisão e *k*-NN ficam no meio: menos fechados que uma reta, menos opacos que uma rede. Os que ainda não apareceram ganham, mais adiante, o capítulo que lhes cabe; o que fica desta seção é a posição relativa, e as duas perguntas a fazer antes de escolher: que pergunta se quer responder, e quantos dados se tem. As duas respostas podem empurrar a escolha para lados opostos do mesmo mapa.

## Supervisionado e Não Supervisionado

> **📌 Nota**
>
> Esta seção corresponde às seções 2.1.4 e 2.1.5 de James et al. (2023).

`Advertising`, `Income1` e `Income2` têm o mesmo formato: preditores que se medem e uma resposta que se quer prever ou explicar (`vendas` num caso, `renda` nos outros dois). Nem todo problema traz essa resposta, e é a presença ou a ausência dela que separa o aprendizado estatístico em duas famílias.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("estilo-figuras.mplstyle")

### Uma resposta para cada observação, ou nenhuma

No aprendizado **supervisionado**, cada observação chega como um par $(x_i, y_i)$, com $i = 1, \dots, n$: os valores dos preditores e a resposta associada a eles. É a resposta que permite conferir o quanto uma estimativa $\hat f$ acerta, e é ela que sustenta as perguntas da seção 7.2: estimar $f$, separar o erro em redutível e irredutível, decidir entre prever e explicar. Sem $y_i$, não sobra nada contra o que comparar a previsão.

No aprendizado **não supervisionado**, cada observação chega só como $x_i$, sem resposta nenhuma. Em muitos problemas reais não existe um $Y$ a registrar: uma loja conhece o que cada cliente comprou, mas não existe um rótulo "verdadeiro" dizendo a que tipo de cliente cada um pertence. A pergunta muda de natureza. Em vez de "o que prevê $Y$", passa a ser "que estrutura existe nestes dados": quantos grupos naturais eles formam, quais variáveis se movem juntas, qual observação foge do padrão das demais.

### A mesma nuvem, duas perguntas

Olhar os mesmos dados das duas formas deixa a diferença clara. A nuvem a seguir é simulada: três grupos de pontos em duas dimensões, cada um sorteado ao redor de um centro diferente.

In [ ]:
rng = np.random.default_rng(7)
n_por_grupo = 40
centros = np.array([[0.0, 0.0], [4.5, 4.0], [-1.0, 5.5]])
X = np.concatenate(
    [rng.normal(loc=centro, scale=1.0, size=(n_por_grupo, 2)) for centro in centros]
)
rotulo = np.repeat(np.arange(len(centros)), n_por_grupo)
X.shape, rotulo.shape

> **🔧 Função**
>
> **`np.array(lista)`**: cria um array a partir de uma lista; uma lista de listas vira uma tabela, uma lista interna por linha. Aqui, `centros` tem uma linha por grupo.
>
> **`np.random.default_rng(semente)`**: cria um gerador de números aleatórios com semente fixa, para que o sorteio saia igual a cada execução.
>
> **`rng.normal(loc, scale, size)`**: sorteia valores de uma normal com média `loc` e desvio padrão `scale`. Aqui, `loc` é o centro do grupo e `size=(40, 2)` pede 40 pontos com duas coordenadas cada.
>
> **`np.concatenate(lista)`**: empilha os arrays da lista, um embaixo do outro.
>
> **`np.arange(n)`** e **`np.repeat(valores, vezes)`**: `np.arange(3)` é `[0, 1, 2]`, e `np.repeat` repete cada valor `vezes` vezes seguidas, aqui 40 zeros, 40 uns e 40 dois, um rótulo para cada ponto.

`X` é um array de 120 linhas e 2 colunas, uma linha por ponto, e `rotulo` guarda, para cada ponto, o grupo que o gerou.

In [ ]:
np.unique(rotulo, return_counts=True)

> **🔧 Função**
>
> **`np.unique(arr, return_counts=True)`**: os valores distintos de `arr`, em ordem, e quantas vezes cada um aparece.

Três grupos, 40 pontos em cada, 120 ao todo. Os dois painéis a seguir mostram a mesma nuvem; muda só o que se sabe sobre ela.

In [ ]:
# Figura: A mesma nuvem simulada, olhada de duas formas. Esquerda: cada ponto colorido pelo grupo que o gerou; com o rótulo conhecido, o problema é supervisionado. Direita: os mesmos 120 pontos, sem cor; sem rótulo, o problema é não supervisionado, e a pergunta passa a ser quantos grupos existem ali.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

for grupo in range(len(centros)):
    pontos_do_grupo = X[rotulo == grupo]
    ax1.scatter(pontos_do_grupo[:, 0], pontos_do_grupo[:, 1], label=f"grupo {grupo}")
ax1.set_title("supervisionado: o rótulo é conhecido")
ax1.set_xlabel("x1")
ax1.set_ylabel("x2")
ax1.legend()

ax2.scatter(X[:, 0], X[:, 1], color="0.5", label="quantos grupos existem aqui?")
ax2.set_title("não supervisionado: sem rótulo")
ax2.set_xlabel("x1")
ax2.set_ylabel("x2")
ax2.legend()

plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`X[rotulo == grupo]`**: `rotulo == grupo` compara cada rótulo com o grupo e devolve um array de `True`/`False`, a **máscara**; indexar `X` com ela seleciona só as linhas onde a máscara vale `True`.
>
> **`X[:, 0]`** e **`X[:, 1]`**: a primeira e a segunda coluna de `X`, em todas as linhas. O `:` antes da vírgula quer dizer "todas as linhas".

À esquerda, a cor de cada ponto vem do grupo que o sorteou. Com esse rótulo em mãos, a pergunta natural é de classificação: dado um ponto novo, com suas coordenadas $x_1$ e $x_2$, a que grupo ele pertence? Há uma resposta certa contra a qual conferir cada previsão. À direita, sem cor nenhuma, sobra a pergunta que dá título à legenda. Um método de agrupamento poderia propor uma partição parecida com a da esquerda, e, olhando a nuvem, três grupos parecem mesmo razoáveis. Mas não há rótulo contra o qual confirmar essa resposta: o que existe é só a estrutura que os pontos sugerem.

### Regressão contra classificação: a natureza de $Y$

O que muda quando $Y$ deixa de ser um número? Dentro do lado supervisionado há uma segunda distinção, que é sobre o tipo de valor que $Y$ assume. Quando $Y$ é **quantitativo**, um número que mede algo (como `vendas` ou `renda`), o problema é de **regressão**, e todos os exemplos do capítulo até aqui foram de regressão. Quando $Y$ é **qualitativo**, uma categoria (como "inadimplente" ou "em dia", "spam" ou "não spam"), o problema é de **classificação**. Não há meio caminho entre duas categorias, e o erro passa a ser contado, acerto ou erro, em vez de medido como distância.

A fronteira separa problemas, mas nem sempre separa métodos. O *k*-NN que a seção 7.3 ajustou a `Income2` encontrou os vizinhos mais próximos de cada ponto e devolveu a média das rendas deles, um número, porque ali $Y$ era quantitativo. Aplicado a um $Y$ qualitativo, o mesmo mecanismo devolve a categoria mais comum entre os vizinhos. A busca pelos vizinhos não muda; muda só o que se faz com eles.

As duas seções seguintes medem a qualidade de um ajuste em cada lado: a 7.6 na regressão, com o erro quadrático médio, e a 7.7 na classificação, com a taxa de erro. O lado não supervisionado ganha, mais adiante, um capítulo próprio para responder com método à pergunta que o painel da direita deixou em aberto.

## Qualidade do Ajuste e o Compromisso Viés-Variância

> **📌 Nota**
>
> Esta seção corresponde às seções 2.2.1 e 2.2.2 de James et al. (2023).

Dois ajustes diferentes aos mesmos dados: qual é melhor? A resposta natural é medir quanto cada um erra e ficar com o que erra menos. Mas **onde** esse erro é medido decide se o número quer dizer alguma coisa.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import make_smoothing_spline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

plt.style.use("estilo-figuras.mplstyle")

### O erro quadrático médio, e a pergunta que interessa

Na regressão, a medida mais usada é o **erro quadrático médio** (MSE, do inglês *mean squared error*). Dado um ajuste $\hat f$ e $n$ observações $(x_i, y_i)$, ele é a média dos quadrados das diferenças entre o observado e o previsto:

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - \hat f(x_i)\right)^2
$$

Se $x_i$ e $y_i$ são os mesmos pontos usados para construir $\hat f$, esse é o **MSE de treino**, e ele engana. Nada impede que $\hat f$ passe exatamente por cada ponto de treino sem ter aprendido nada sobre a relação entre $X$ e $Y$, só decorando as respostas que já tinha. É o *overfitting*, o ajuste que segue o ruído de uma amostra em vez da relação que a gerou. Um MSE de treino baixo não distingue um ajuste que aprendeu a relação de um que decorou a amostra.

A pergunta que interessa é outra: quão perto $\hat f$ chega de um ponto que ele **nunca viu**? O **MSE de teste**, medido em observações que ficaram de fora do ajuste, responde a ela. Ele estima o erro esperado inteiro da seção 7.2, a parte redutível mais a irredutível, $\mathrm{Var}(\varepsilon)$. Como a parte irredutível é a mesma para qualquer método, comparar o MSE de teste de dois ajustes é comparar o quanto cada um deixa de erro redutível.

### Separando treino e teste: `train_test_split`

Para ter um conjunto de teste, basta separá-lo antes de ajustar qualquer coisa: uma fração dos dados fica reservada como **teste**, sem nunca entrar em ajuste nenhum, e o resto vira **treino**. É essa separação que permite julgar um ajuste pelo erro em dados novos.

Para ver o que a flexibilidade de um ajuste tem de real, e o que é só o ajuste seguindo o ruído, ainda falta uma peça: conhecer a própria $f$. Com dados reais isso é impossível. Com dados **simulados**, gerados por uma função escolhida por quem simula, a $f$ verdadeira é conhecida número por número.

In [ ]:
def f_verdadeiro(x):
    return 4.0 + 0.3 * x + 3.0 * np.sin(x)

rng = np.random.default_rng(7)
n = 300
ruido_padrao = 1.5

x = np.sort(rng.uniform(0, 10, size=n))
ruido = rng.normal(0, ruido_padrao, size=n)
y = f_verdadeiro(x) + ruido

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y, test_size=0.3, random_state=7
)
x_treino.shape, x_teste.shape

> **🔧 Função**
>
> **`rng.uniform(inicio, fim, size)`**: sorteia `size` valores uniformes entre `inicio` e `fim` (entre 0 e 1, sem os dois primeiros argumentos). O `rng` e o `rng.normal` são os mesmos da seção 7.5.
>
> **`np.sort(arr)`**: devolve os valores de `arr` em ordem crescente.
>
> **`train_test_split(x, y, test_size, random_state)`**: sorteia quais observações vão para o teste e devolve, nesta ordem, os preditores de treino, os de teste, as respostas de treino e as de teste.
>
> - `test_size=0.3`: 30% das observações vão para o teste.
> - `random_state=7`: a semente do sorteio, para que a divisão saia sempre a mesma.

Trezentos pontos gerados por uma função conhecida, uma tendência linear somada a uma oscilação de seno, mais um ruído normal com desvio padrão `ruido_padrao = 1.5`. A divisão deixa 210 pontos de treino e 90 de teste. As duas sementes, a do `rng` e a do `random_state`, fixam sorteios diferentes, e as duas precisam estar fixas para a figura sair sempre igual.

`ruido_padrao` é o desvio padrão de $\varepsilon$, e portanto $\mathrm{Var}(\varepsilon) = 1{,}5^2 = 2{,}25$: o piso irredutível da seção 7.2, que aqui se conhece porque foi escolhido na simulação.

### Três ajustes, três flexibilidades

Com a $f$ verdadeira conhecida, dá para comparar ajustes de rigidez bem diferente contra ela, e não só contra os dados. O mais rígido é uma reta, com `LinearRegression`. Os outros dois são *smoothing splines*: curvas suaves ajustadas minimizando a soma dos quadrados dos resíduos mais uma penalidade sobre a curvatura, com o peso da penalidade dado por um parâmetro $\lambda$. Quanto maior $\lambda$, mais a curvatura é punida e mais a curva se aproxima de uma reta; quanto menor $\lambda$, mais livre a curva fica para se dobrar atrás de cada ponto.

In [ ]:
ordem = np.argsort(x_treino)
x_treino_ordenado = x_treino[ordem]
y_treino_ordenado = y_treino[ordem]

reta = LinearRegression().fit(x_treino.reshape(-1, 1), y_treino)
spline_moderado = make_smoothing_spline(x_treino_ordenado, y_treino_ordenado, lam=0.5)
spline_flexivel = make_smoothing_spline(
    x_treino_ordenado, y_treino_ordenado, lam=0.0001
)

resultado = pd.DataFrame(
    {
        "treino": [
            mean_squared_error(y_treino, reta.predict(x_treino.reshape(-1, 1))),
            mean_squared_error(y_treino, spline_moderado(x_treino)),
            mean_squared_error(y_treino, spline_flexivel(x_treino)),
        ],
        "teste": [
            mean_squared_error(y_teste, reta.predict(x_teste.reshape(-1, 1))),
            mean_squared_error(y_teste, spline_moderado(x_teste)),
            mean_squared_error(y_teste, spline_flexivel(x_teste)),
        ],
    },
    index=["reta", "spline moderado (λ=0,5)", "spline muito flexível (λ=0,0001)"],
)
resultado.round(2)

> **🔧 Função**
>
> **`np.argsort(arr)`**: as posições que põem `arr` em ordem crescente. `x_treino[ordem]` reordena o array por essas posições; a mesma `ordem` aplicada a `y_treino` mantém cada $y$ junto do seu $x$.
>
> **`arr.reshape(-1, 1)`**: transforma um array de uma dimensão numa tabela de uma coluna só. O `LinearRegression` pede os preditores nesse formato, uma coluna por preditor; o `-1` quer dizer "quantas linhas forem necessárias".
>
> **`make_smoothing_spline(x, y, lam)`**: ajusta um *smoothing spline* aos pontos, que precisam vir com `x` em ordem crescente. `lam` é o $\lambda$ da penalidade. Devolve uma função: `spline_moderado(x_teste)` dá a previsão em cada ponto de `x_teste`.

In [ ]:
resultado["teste"].idxmin(), resultado["treino"].is_monotonic_decreasing

> **🔧 Função**
>
> **`serie.is_monotonic_decreasing`**: `True` se cada valor é menor ou igual ao anterior.

In [ ]:
# Figura: Trezentos pontos simulados a partir de uma f conhecida (azul), divididos em treino (círculos cinza) e teste (triângulos cinza), com três ajustes de flexibilidade crescente. A reta mal acompanha a curva; o spline muito flexível se dobra atrás de cada ponto de treino, copiando também o ruído de cada um.
grade = np.linspace(x.min(), x.max(), 400)

fig, ax = plt.subplots()
ax.scatter(x_treino, y_treino, s=14, alpha=0.5, color="#7A8894", label="treino")
ax.scatter(
    x_teste, y_teste, s=20, alpha=0.6, color="#7A8894", marker="^", label="teste"
)
ax.plot(grade, f_verdadeiro(grade), color="C0", linewidth=2.5, label="f verdadeira")
previsao_reta = reta.predict(grade.reshape(-1, 1))
ax.plot(grade, previsao_reta, color="C1", linewidth=2, label="reta")
ax.plot(
    grade, spline_moderado(grade), color="C2", linewidth=2, label="spline moderado"
)
ax.plot(
    grade, spline_flexivel(grade), color="C3", linewidth=2,
    label="spline muito flexível",
)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(0, 10)
ax.legend(ncols=2)
plt.tight_layout()
plt.show()

A reta erra 5,83 de MSE no treino e 5,28 no teste. Rígida demais para acompanhar a oscilação do seno, ela erra parecido nos dois conjuntos, porque a forma que assumiu está errada em qualquer amostra. O spline muito flexível faz o oposto: 1,05 no treino, quase decorando cada ponto, contra 2,77 no teste. O spline moderado tem o menor MSE de teste dos três (`idxmin` aponta para ele), com 1,66 no treino e 2,31 no teste.

A comparação da segunda saída mostra o formato: o MSE de treino só cai conforme a flexibilidade cresce (`is_monotonic_decreasing` é `True`: 5,83, 1,66, 1,05), enquanto o de teste cai e depois sobe (5,28, 2,31, 2,77). Treino sempre caindo, teste em U: a varredura a seguir mostra esse formato com muito mais que três pontos.

### A curva em U, e o piso que ela não fura

Uma varredura em $\lambda$, do mais rígido ao mais flexível, desenha a curva inteira. Só que o MSE de teste, medido sobre poucos pontos, também varia com o sorteio: com só 90 pontos, ele pode cair abaixo do piso irredutível por sorte da amostra, sem que o piso tenha sido furado de verdade. Como a simulação dá acesso a $f$, dá para gerar um conjunto de teste muito maior, da mesma $f$ e com o mesmo ruído.

In [ ]:
n_teste_grande = 5000
x_teste_grande = rng.uniform(0, 10, size=n_teste_grande)
ruido_teste_grande = rng.normal(0, ruido_padrao, size=n_teste_grande)
y_teste_grande = f_verdadeiro(x_teste_grande) + ruido_teste_grande
x_teste_grande.shape

Cinco mil pontos novos, que não entram em ajuste nenhum e servem só para medir a varredura.

In [ ]:
lambdas = np.logspace(3, -5, 60)
mse_treino = np.array(
    [
        mean_squared_error(
            y_treino, make_smoothing_spline(x_treino_ordenado, y_treino_ordenado, lam=l)(x_treino)
        )
        for l in lambdas
    ]
)
mse_teste = np.array(
    [
        mean_squared_error(
            y_teste_grande,
            make_smoothing_spline(x_treino_ordenado, y_treino_ordenado, lam=l)(x_teste_grande),
        )
        for l in lambdas
    ]
)

piso = ruido_padrao**2
indice_minimo = int(np.argmin(mse_teste))
lambda_minimo = lambdas[indice_minimo]
mse_teste_minimo = mse_teste[indice_minimo]

treino_sempre_cai = bool(np.all(np.diff(mse_treino) <= 0))
teste_nunca_fura_piso = bool(np.all(mse_teste >= piso))

piso, round(float(lambda_minimo), 2), round(float(mse_teste_minimo), 2), treino_sempre_cai, teste_nunca_fura_piso

> **🔧 Função**
>
> **`np.logspace(a, b, n)`**: `n` números igualmente espaçados em escala logarítmica, de $10^a$ a $10^b$. Aqui, 60 valores de $\lambda$ de 1.000 a 0,00001.
>
> **`np.argmin(arr)`**: a posição do menor valor de `arr`.
>
> **`np.diff(arr)`**: a diferença entre cada elemento e o anterior. `np.all(np.diff(mse_treino) <= 0)` pergunta se nenhuma diferença é positiva, isto é, se o MSE de treino nunca sobe.
>
> **`np.all(mascara)`**: `True` se todos os elementos da máscara são `True`.

In [ ]:
# Figura: MSE de treino e de teste (este sobre os 5.000 pontos do conjunto de teste grande) contra a flexibilidade do spline (λ decrescente, escala log). O treino cai sem parar; o teste cai, toca um mínimo perto de λ=0,30 e volta a subir. A linha tracejada é Var(ε), o piso irredutível, conhecido porque o ruído foi gerado na simulação; a curva de teste nunca desce abaixo dela.
fig, ax = plt.subplots()
ax.plot(lambdas, mse_treino, linewidth=2, label="MSE treino")
ax.plot(lambdas, mse_teste, linewidth=2, label="MSE teste")
ax.axhline(piso, color="0.45", linestyle="--", linewidth=1.5, label="piso irredutível (Var(ε))")
ax.set_xscale("log")
ax.set_xlim(lambdas.max(), lambdas.min())
ax.set_xlabel("λ do spline (escala log; menor λ = mais flexível →)")
ax.set_ylabel("MSE")
ax.legend()
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.axhline(y, color, linestyle)`**: uma linha horizontal na altura `y`, de uma ponta à outra do gráfico; `color="0.45"` é um cinza.
>
> **`ax.set_xscale("log")`**: põe o eixo horizontal em escala logarítmica. Com o `set_xlim` indo do maior $\lambda$ ao menor, a flexibilidade cresce da esquerda para a direita.

Ao longo dos 60 valores de $\lambda$, o MSE de treino nunca sobe (`treino_sempre_cai` é `True`). O de teste desce a partir do extremo rígido, toca o mínimo de 2,41 perto de $\lambda = 0{,}30$ e volta a subir do outro lado, onde a flexibilidade passa a seguir o ruído. O mínimo fica perto do spline moderado ($\lambda = 0{,}5$), escolhido às cegas na comparação anterior, sem coincidir com ele. Em nenhum dos 60 pontos o MSE de teste desce abaixo de `piso = 2.25` (`teste_nunca_fura_piso` é `True`). É o piso da seção 7.2, agora desenhado: nenhuma flexibilidade põe o erro esperado abaixo da parte do problema que o ajuste não controla.

### A decomposição viés-variância

Por que a curva de teste tem forma de U? Imagine repetir o experimento muitas vezes: sortear um conjunto de treino novo, ajustar $\hat f$ nele e prever num ponto novo $x_0$, cuja resposta é $y_0 = f(x_0) + \varepsilon$. A cada repetição, $\hat f(x_0)$ sai um pouco diferente. A média do erro quadrático sobre todas essas repetições (sobre os conjuntos de treino possíveis e sobre o ruído de $y_0$) se decompõe em três parcelas:

$$
\mathrm{E}\left[\left(y_0 - \hat f(x_0)\right)^2\right] = \mathrm{Var}\left(\hat f(x_0)\right) + \left[\mathrm{Bias}\left(\hat f(x_0)\right)\right]^2 + \mathrm{Var}(\varepsilon)
$$

O **viés** (*bias*) é o erro que vem de a forma escolhida ser rígida demais para a relação verdadeira: o preço que a reta paga por ser reta, mesmo com todos os dados de treino do mundo. A **variância** é o quanto $\hat f(x_0)$ muda de um conjunto de treino para outro: o preço que o spline muito flexível paga por seguir de perto os pontos que calhou de receber. Um pouco de ruído a mais ou a menos nesses pontos, e a curva inteira se reacomoda. $\mathrm{Var}(\varepsilon)$ é o piso, que nenhuma das duas outras parcelas toca.

Os três ajustes ocupam lugares diferentes nessa conta. A reta tem viés alto, porque nenhuma reta acompanha a curvatura do seno, e variância baixa, porque trocar a amostra de treino move pouco uma reta. O spline muito flexível inverte os dois: viés baixo, porque consegue seguir qualquer curvatura, e variância alta, porque essa liberdade o deixa refém do ruído da amostra. O spline moderado não zera nenhuma das duas parcelas, mas entre os três é o que tem o menor MSE de teste, que é a estimativa dessa soma.

A decomposição vale num ponto $x_0$. O MSE de teste esperado é a média dela sobre todos os pontos do conjunto de teste. A curva desenhada acima é uma única realização disso: um conjunto de treino, um ajuste por $\lambda$. Ela acompanha a soma viés² + variância + $\mathrm{Var}(\varepsilon)$ sem ser exatamente essa soma, que é uma média sobre muitos conjuntos de treino. Mas o formato é o mesmo. Conforme a flexibilidade cresce, o viés cai e a variância sobe. No começo o viés cai mais depressa, e o erro de teste desce. Depois a variância passa a dominar, e ele sobe.

## Classificação e o Classificador de Bayes

> **📌 Nota**
>
> Esta seção corresponde à seção 2.2.3 de James et al. (2023).

Quando $Y$ é uma classe (doente ou não, spam ou não), a distância entre a previsão e o valor observado deixa de fazer sentido. A régua muda: o que conta é se a previsão acertou o rótulo ou errou.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

plt.style.use("estilo-figuras.mplstyle")

### A taxa de erro

Para uma resposta qualitativa, o análogo do MSE é a **taxa de erro**, a fração das previsões que erram o rótulo:

$$
\text{Taxa de erro} = \frac{1}{n}\sum_{i=1}^n I(y_i \neq \hat y_i)
$$

$\hat y_i$ é a classe prevista para a observação $i$, e $I(y_i \neq \hat y_i)$ é uma **variável indicadora**: vale 1 quando o classificador erra a observação $i$ e 0 quando acerta. A média sobre as $n$ observações é a proporção de erros. Vale a mesma regra do MSE: a taxa de erro que interessa é a **de teste**, medida em observações que não foram usadas para ajustar o classificador. A de **treino** diz o quanto ele decorou o que já tinha visto.

### O classificador de Bayes

Se fosse possível saber, para cada ponto, a probabilidade de cada classe, qual regra de previsão erraria menos? A resposta é simples de enunciar: em cada ponto $x_0$, prever a classe mais provável dado $X = x_0$.

$$
\hat C(x_0) = \operatorname*{argmax}_{j} \; \Pr(Y = j \mid X = x_0)
$$

$\Pr(Y = j \mid X = x_0)$ é a probabilidade condicional de a classe ser $j$ para quem tem $X = x_0$. Prever qualquer outra classe em $x_0$ só aumenta a chance de errar ali, e por isso nenhuma regra faz melhor. Essa regra se chama **classificador de Bayes**, e a taxa de erro dela, **taxa de erro de Bayes**: a menor taxa de erro de teste esperada que um classificador pode ter, análogo ao $\mathrm{Var}(\varepsilon)$ da regressão.

Descrever o classificador de Bayes é mais fácil que construí-lo. Ele exige conhecer $\Pr(Y = j \mid X = x_0)$ para todo $x_0$, e com dados reais ninguém tem essa distribuição: o que existe são $n$ observações. Por isso o classificador de Bayes é inatingível na prática, um padrão contra o qual comparar os outros. Com dados simulados, porém, é quem simula que escolhe $\Pr(Y \mid X)$, e aí o piso pode ser calculado.

In [ ]:
def p_verdadeiro(x1, x2):
    return 1.0 / (1.0 + np.exp(-(3.0 * np.sin(x1) - x2 + 5.0)))

rng = np.random.default_rng(7)

def gerar(rng, n):
    x1 = rng.uniform(0, 10, size=n)
    x2 = rng.uniform(0, 10, size=n)
    p = p_verdadeiro(x1, x2)
    y = (rng.uniform(size=n) < p).astype(int)
    return np.column_stack([x1, x2]), y

X_treino, y_treino = gerar(rng, 300)
X_teste, y_teste = gerar(rng, 20_000)
X_treino.shape, X_teste.shape

> **🔧 Função**
>
> **`np.exp(arr)`** e **`np.sin(arr)`**: a exponencial e o seno de cada elemento.
>
> **`mascara.astype(int)`**: converte `True`/`False` em 1/0.
>
> **`np.column_stack([a, b])`**: junta dois arrays de uma dimensão como as colunas de uma tabela.

Dois preditores, $X_1$ e $X_2$, uniformes em $[0, 10]$. A probabilidade de $Y = 1$ é uma logística sobre uma combinação deles, uma tendência linear em $x_2$ mais uma oscilação de seno em $x_1$:

$$
\Pr(Y = 1 \mid X = x) = \frac{1}{1 + e^{-(3\sin(x_1) - x_2 + 5)}}
$$

O rótulo sai de um sorteio com essa probabilidade: `rng.uniform(size=n) < p` vale `True` com probabilidade $p$, e isso é um sorteio de Bernoulli$(p)$. São 300 pontos de treino e 20.000 de teste; o motivo de tantos pontos de teste aparece quando o *k*-NN for comparado com o piso.

A **fronteira de Bayes** é onde $\Pr(Y = 1 \mid X = x) = 1/2$ e as duas classes empatam. Ela fica onde o expoente da logística zera, $x_2 = 3\sin(x_1) + 5$, uma curva. Fora dela, uma classe é sempre mais provável que a outra, mas nunca com certeza: em todo ponto do quadrado, a probabilidade fica estritamente entre 0 e 1. Há sempre alguma chance de a classe menos provável aparecer, e é essa sobreposição que faz a taxa de erro de Bayes ser maior que zero.

In [ ]:
grade_integral = np.linspace(0, 10, 4001)
G1, G2 = np.meshgrid(grade_integral, grade_integral)
Pg = p_verdadeiro(G1, G2)
piso_bayes = float(np.mean(np.minimum(Pg, 1 - Pg)))

pred_bayes_teste = (p_verdadeiro(X_teste[:, 0], X_teste[:, 1]) > 0.5).astype(int)
erro_bayes_teste = float(np.mean(pred_bayes_teste != y_teste))

round(piso_bayes, 4), round(erro_bayes_teste, 4)

> **🔧 Função**
>
> **`np.minimum(a, b)`**: o menor dos dois valores, elemento a elemento.
>
> **`np.mean(previsto != observado)`**: a comparação dá um array de `True`/`False`, e a média dele é a fração de `True`, isto é, a taxa de erro.

A taxa de erro de Bayes pode ser calculada sem sorteio nenhum. Em cada ponto $x$, o classificador de Bayes erra com a chance da classe que ele não escolheu:

$$
\min\big(\Pr(Y{=}1\mid X{=}x),\; 1 - \Pr(Y{=}1\mid X{=}x)\big)
$$

Como $X$ é uniforme no quadrado $[0, 10]^2$, a taxa de erro de Bayes é a média dessa quantidade sobre o quadrado, e numa grade fina ela dá **13,26%**. Aplicar a regra de Bayes aos 20.000 pontos de teste, prevendo a classe mais provável em cada um e comparando com o $y$ que saiu do sorteio, dá **13,32%**: quase o mesmo número, com a diferença que se espera de um sorteio finito.

### *k* vizinhos mais próximos

E quando $\Pr(Y \mid X)$ é desconhecida, como com dados reais? O classificador de Bayes conhece essa probabilidade; o *k*-NN não conhece, e estima. Para um ponto $x_0$, ele olha os $k$ pontos de treino mais próximos e prevê a classe que aparece mais entre eles. A fração de vizinhos da classe 1 é uma estimativa de $\Pr(Y{=}1\mid X{=}x_0)$, feita localmente, com $k$ pontos.

$k$ decide o quanto essa estimativa é local. Com $k$ pequeno, a vizinhança tem poucos pontos, às vezes um só, e a fronteira segue cada observação de treino: variância alta, porque trocar a amostra de treino move a fronteira inteira. Com $k$ grande, pontos distantes de $x_0$ entram na votação, diluem a estrutura local, e a fronteira endurece: viés alto. No limite, com $k$ perto do número de pontos de treino, todo ponto tem praticamente os mesmos vizinhos, e o classificador prevê a mesma classe, a mais comum no treino, em todo lugar.

In [ ]:
ks = list(range(1, 300, 4))
erros_treino = []
erros_teste = []
for k in ks:
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_treino, y_treino)
    previsto_treino = modelo.predict(X_treino)
    previsto_teste = modelo.predict(X_teste)
    erros_treino.append(
        float(np.mean(previsto_treino != y_treino))
    )
    erros_teste.append(
        float(np.mean(previsto_teste != y_teste))
    )

erros_treino = np.array(erros_treino)
erros_teste = np.array(erros_teste)
indice_minimo = int(np.argmin(erros_teste))
k_minimo = ks[indice_minimo]
teste_minimo = float(erros_teste[indice_minimo])
nunca_abaixo_do_piso = bool(np.all(erros_teste >= piso_bayes))
margem_pp = round((teste_minimo - piso_bayes) * 100, 2)

len(ks), ks[0], ks[-1], k_minimo, round(teste_minimo, 4), round(float(erros_treino[0]), 4), nunca_abaixo_do_piso, margem_pp

> **🔧 Função**
>
> **`KNeighborsClassifier(n_neighbors)`**: cria um *k*-NN para resposta qualitativa, que prevê a classe mais comum entre os `n_neighbors` vizinhos mais próximos. Ajusta-se com `.fit(X, y)` e prevê com `.predict(X)`, como os modelos de regressão.

A varredura cobre 75 valores de $k$, de 1 a 297, de quatro em quatro. Com $k = 1$, o erro de treino é **0%**: cada ponto de treino é o seu próprio vizinho mais próximo, e a votação sempre devolve o rótulo que ele já tinha. O menor erro de teste sai em $k = 9$: **15,08%**. Em nenhum dos 75 valores a taxa de erro de teste desce abaixo dos 13,26% do piso de Bayes; mesmo o melhor, $k = 9$, fica **1,82** ponto percentual acima (`margem_pp`). Com poucas centenas de pontos de teste, uma distância desse tamanho poderia sumir por sorte da amostra; com 20.000, a comparação não depende do sorteio.

Uma ressalva sobre esse $k = 9$: ele foi escolhido olhando o próprio conjunto de teste, e por isso os 15,08% saem um pouco otimistas. Aqui isso serve para desenhar a curva inteira; na prática, o $k$ é escolhido sem tocar no teste, com a validação cruzada do capítulo 10.

A figura a seguir põe no eixo horizontal $1/k$, que cresce com a flexibilidade: $k$ pequeno fica à direita.

In [ ]:
# Figura: Taxa de erro de treino e de teste do k-NN contra 1/k (escala log), da vizinhança maior (esquerda) à menor (direita). O treino chega a 0% em k=1; o teste desenha um U, com o mínimo em k=9, e nunca desce abaixo da taxa de erro de Bayes (tracejada).
inverso_k = 1 / np.array(ks)

fig, ax = plt.subplots()
ax.plot(inverso_k, erros_treino, linewidth=2, label="erro de treino")
ax.plot(inverso_k, erros_teste, linewidth=2, label="erro de teste")
ax.axhline(piso_bayes, color="0.45", linestyle="--", linewidth=1.5, label="taxa de erro de Bayes")
ax.set_xscale("log")
ax.set_xlim(inverso_k.min() * 0.8, 1.25)
ax.set_ylim(0, None)
ax.yaxis.set_major_formatter(lambda valor, _: f"{valor:.0%}")
ax.set_xlabel("1/k (escala log; mais flexível →)")
ax.set_ylabel("taxa de erro")
ax.legend()
plt.tight_layout()
plt.show()

A figura tem uma surpresa: em boa parte da faixa de $k$, o erro de treino fica **acima** do de teste, ao contrário do que se espera de um erro medido nos próprios pontos do ajuste.

In [ ]:
treino_acima = int(np.sum(erros_treino > erros_teste))
bayes_no_treino = np.mean(
    (p_verdadeiro(X_treino[:, 0], X_treino[:, 1]) > 0.5).astype(int) != y_treino
)
treino_acima, round(float(bayes_no_treino), 4)

O erro de treino passa do de teste em 58 dos 75 valores de $k$. O erro de treino é medido em só 300 pontos, e uma taxa medida em 300 pontos oscila alguns pontos percentuais de uma amostra para outra; a de teste, medida em 20.000, quase não oscila. E como os 75 valores de $k$ usam os mesmos 300 pontos, as taxas de treino oscilam em boa parte juntas, sobretudo para valores de $k$ próximos. Nesta amostra, a oscilação foi para cima: até a regra de Bayes, que é a de menor erro esperado, erra 16,33% dos pontos de treino, contra os 13,26% que erra no quadrado inteiro. O que a figura ensina é o formato das curvas (treino caindo até zero com $k$ pequeno, teste em U), e não a posição de uma em relação à outra.

### A fronteira que cada $k$ desenha

Três valores de $k$ bem separados mostram o que a varredura contou em número: $k = 1$, o mínimo ($k = 9$) e $k = 199$, mais perto do outro extremo. Além dos erros, o chunk mede o quanto cada fronteira discorda da de Bayes: a fração de uma grade sobre o quadrado em que o *k*-NN e o classificador de Bayes preveem classes diferentes.

In [ ]:
k_grande = 199
k_tres = [1, k_minimo, k_grande]
modelos_tres = [
    KNeighborsClassifier(n_neighbors=k).fit(X_treino, y_treino) for k in k_tres
]

grade_fig = np.linspace(0, 10, 200)
Xg, Yg = np.meshgrid(grade_fig, grade_fig)
pontos_grade = np.column_stack([Xg.ravel(), Yg.ravel()])
bayes_grade = (p_verdadeiro(pontos_grade[:, 0], pontos_grade[:, 1]) > 0.5).astype(int)

tabela_k = pd.DataFrame(
    {
        "treino": [float(np.mean(m.predict(X_treino) != y_treino)) for m in modelos_tres],
        "teste": [float(np.mean(m.predict(X_teste) != y_teste)) for m in modelos_tres],
        "discorda de Bayes": [
            float(np.mean(m.predict(pontos_grade) != bayes_grade)) for m in modelos_tres
        ],
    },
    index=[f"k={k_tres[0]}", f"k={k_tres[1]} (menor erro de teste)", f"k={k_tres[2]}"],
)
tabela_k.round(4)

In [ ]:
tabela_k["teste"].idxmin(), tabela_k["discorda de Bayes"].idxmin()

$k = 1$ decora o treino (0% de erro) e erra 20,92% do teste. $k = 199$ erra 26,67% do treino e 23,03% do teste, rígido demais para acompanhar a curva $x_2 = 3\sin(x_1) + 5$. $k = 9$ fica entre os dois no treino (14,33%) e tem o menor erro de teste dos três (15,08%, a primeira saída do `idxmin`). A última coluna conta a mesma história pela fronteira: $k = 9$ discorda da regra de Bayes em 7,77% da grade, contra 15,70% de $k = 1$ e 19,45% de $k = 199$, e é o que menos discorda dos três (a segunda saída do `idxmin`).

In [ ]:
# Figura: Fronteiras de decisão do k-NN para k=1, k=9 e k=199 (verde), sobre os mesmos 300 pontos de treino coloridos pela classe verdadeira, com a fronteira de Bayes (roxo tracejado) por cima. k=1 se dobra atrás de cada ponto; k=199 quase não acompanha a curvatura da fronteira verdadeira; k=9 é o que discorda menos dela.
proba_bayes = p_verdadeiro(Xg, Yg)

fig, eixos = plt.subplots(1, 3, figsize=(12, 4.6), sharex=True, sharey=True)
for ax, k, modelo in zip(eixos, k_tres, modelos_tres):
    proba_knn = modelo.predict_proba(pontos_grade)[:, 1].reshape(Xg.shape)
    ax.scatter(
        X_treino[y_treino == 0, 0], X_treino[y_treino == 0, 1],
        s=12, alpha=0.6, color="#4195D1", label="classe 0",
    )
    ax.scatter(
        X_treino[y_treino == 1, 0], X_treino[y_treino == 1, 1],
        s=12, alpha=0.6, color="#D9480F", label="classe 1",
    )
    ax.contour(Xg, Yg, proba_knn, levels=[0.5], colors="#2F8F46", linewidths=2)
    ax.contour(Xg, Yg, proba_bayes, levels=[0.5], colors="#9775FA", linestyles="--", linewidths=1.5)
    ax.set_title(f"k = {k}")
    ax.set_xlabel("x1")
eixos[0].set_ylabel("x2")
eixos[0].plot([], [], color="#2F8F46", linewidth=2, label="fronteira k-NN")
eixos[0].plot([], [], color="#9775FA", linestyle="--", linewidth=1.5, label="fronteira de Bayes")
alcas, rotulos = eixos[0].get_legend_handles_labels()
fig.legend(alcas, rotulos, loc="lower center", ncols=4, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

> **🔧 Função**
>
> **`modelo.predict_proba(X)`**: em vez da classe, a probabilidade estimada de cada classe para cada linha de `X`, uma coluna por classe, na ordem de `modelo.classes_`. Aqui as classes são 0 e 1, e `[:, 1]` separa a probabilidade da classe 1.
>
> **`X_treino[y_treino == 0, 0]`**: `y_treino == 0` é uma máscara de `True`/`False`, uma por linha; antes da vírgula ela escolhe as linhas da classe 0, e o `0` depois da vírgula escolhe a primeira coluna.
>
> **`ax.contour(x, y, z, levels)`**: desenha as curvas onde `z` vale cada número de `levels`. Com `levels=[0.5]`, a curva onde a probabilidade da classe 1 é meio: a fronteira de decisão.
>
> **`fig.legend(alcas, rotulos, loc, ncols)`**: uma legenda só para a figura inteira, fora dos painéis.

É o mesmo compromisso da regressão, medido em taxa de erro em vez de MSE. $k$ pequeno decora o treino e erra o teste por variância. $k$ grande simplifica demais e erra por viés. O meio-termo, aqui $k = 9$, erra menos no teste sem descer abaixo do piso de Bayes. Na classificação, a taxa de erro não se decompõe exatamente numa soma de viés ao quadrado e variância, como o MSE, mas o formato do compromisso, com a curva de teste em U, é o mesmo.

## Leituras adicionais

- O [site oficial de James et al. (2023)](https://www.statlearning.com), com o PDF gratuito do livro, os dados usados neste capítulo e os laboratórios em Python.
- [*NumPy: the absolute basics for beginners*](https://numpy.org/doc/stable/user/absolute_beginners.html), a introdução oficial ao array que a seção 7.1 apresenta.
- [*Copies and views*](https://numpy.org/doc/stable/user/basics.copies.html), o guia oficial sobre a diferença entre fatiar um array e copiá-lo, o que a seção 7.1 chama de "a vista".

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.